In [12]:
import tensorflow as tf
tf.random.set_seed(42)

In [13]:
import sys
sys.path.append("../src")

from data_loader import load_all
from lstm_utils import build_sequences
from fault_reference import HARD_TO_DETECT_FAULTS

train, test = load_all()

measurement_cols = [c for c in train.columns if c.startswith("XMEAS") or c.startswith("XMV")]

train_filtered = train[~train["faultNumber"].isin(HARD_TO_DETECT_FAULTS)]

X_train_seq, y_train_seq = build_sequences(train_filtered, measurement_cols, window=20)

print("X shape:", X_train_seq.shape)
print("y shape:", y_train_seq.shape)

X shape: (8779, 20, 52)
y shape: (8779,)


In [14]:
#test data
test_filtered = test[~test["faultNumber"].isin(HARD_TO_DETECT_FAULTS)]

X_test_seq, y_test_seq = build_sequences(test_filtered, measurement_cols, window=20)

print("X_test shape:", X_test_seq.shape)
print("y_test shape:", y_test_seq.shape)

X_test shape: (17879, 20, 52)
y_test shape: (17879,)


In [15]:
# standardizing and encodeing the fault labels. Standardization is necessary for lstm models to perform well. 
# The fault labels are encoded to integers for the model to predict.    

from sklearn.preprocessing import StandardScaler, LabelEncoder

n_samples, n_timesteps, n_features = X_train_seq.shape

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_seq.reshape(-1, n_features))
X_train_scaled = X_train_scaled.reshape(n_samples, n_timesteps, n_features)

n_test_samples = X_test_seq.shape[0]
X_test_scaled = scaler.transform(X_test_seq.reshape(-1, n_features))
X_test_scaled = X_test_scaled.reshape(n_test_samples, n_timesteps, n_features)

label_encoder_lstm = LabelEncoder()
y_train_encoded = label_encoder_lstm.fit_transform(y_train_seq)
y_test_encoded = label_encoder_lstm.transform(y_test_seq)

print("X_train_scaled shape:", X_train_scaled.shape)
print("Number of classes:", len(label_encoder_lstm.classes_))

X_train_scaled shape: (8779, 20, 52)
Number of classes: 19


In [16]:
# Building the LSTM model using Keras. The model consists of two LSTM layers \
# followed by a dense layer with softmax activation for multi-class classification.

from tensorflow import keras
from tensorflow.keras import layers

model_lstm = keras.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(64),
    layers.Dense(19, activation="softmax")
])

model_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_lstm.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 64)             │        29,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 19)             │         1,235 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,187 (121.82 KB)

 Trainable params: 31,187 (121.82 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
#Training the LSTM model using the training data. The model is trained for 20 epochs with a batch size of 64. 
# A validation split of 0.2 is used to monitor the model's performance on unseen data during training.

history = model_lstm.fit(
    X_train_scaled, y_train_encoded,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 18s 76ms/step - accuracy: 0.5462 - loss: 1.6658 - val_accuracy: 0.0000e+00 - val_loss: 5.6175
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - accuracy: 0.8502 - loss: 0.5126 - val_accuracy: 0.0000e+00 - val_loss: 7.2590
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.9489 - loss: 0.2204 - val_accuracy: 0.0034 - val_loss: 7.9247
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - accuracy: 0.9806 - loss: 0.1021 - val_accuracy: 0.0074 - val_loss: 8.2736
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.9935 - loss: 0.0480 - val_accuracy: 0.0120 - val_loss: 8.2330
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 7s 59ms/step - accuracy: 0.9957 - loss: 0.0315 - val_accuracy: 0.0137 - val_loss: 8.5234
Epoch 7/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - accuracy: 0.9966 - loss: 0.0226 - val_accuracy: 0.0148 - val_loss: 8.5933
Epoch 8/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - accuracy: 0.9920 - loss: 0.0386 -

`validation_split=0.2` doesn't shuffle — it just takes the last 20% of
the array. Since sequences are built fault-by-fault in order, that
last 20% was mostly just the final few faults, which training barely
saw. Result: 100% train accuracy, ~0% validation (below random chance)
— a broken split, not real overfitting.

**Fix**: `train_test_split(..., stratify=...)` first, then pass
`validation_data=(...)` explicitly instead.

In [18]:
# Properly splitting the data and retraining

from sklearn.model_selection import train_test_split

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_scaled, y_train_encoded,
    test_size=0.2,
    stratify=y_train_encoded,
    random_state=42
)

history = model_lstm.fit(
    X_train_split, y_train_split,
    validation_data=(X_val_split, y_val_split),
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - accuracy: 0.8550 - loss: 0.5234 - val_accuracy: 0.9288 - val_loss: 0.2308
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - accuracy: 0.9694 - loss: 0.1182 - val_accuracy: 0.9880 - val_loss: 0.0617
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - accuracy: 0.9825 - loss: 0.0812 - val_accuracy: 0.9880 - val_loss: 0.0541
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.9977 - loss: 0.0242 - val_accuracy: 0.9983 - val_loss: 0.0160
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - accuracy: 1.0000 - loss: 0.0076 - val_accuracy: 0.9994 - val_loss: 0.0113
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 1.0000 - loss: 0.0048 - val_accuracy: 0.9994 - val_loss: 0.0093
Epoch 7/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 49ms/step - accuracy: 1.0000 - loss: 0.0034 - val_accuracy: 0.9994 - val_loss: 0.0084
Epoch 8/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - accuracy: 1.0000 - loss: 0.0027 - val_acc

## Fixing data leakage from overlapping windows

99.77% validation accuracy was suspiciously high — likely because
`step=1` creates heavily overlapping sequences (95% shared rows
between adjacent windows). Random shuffling before splitting let
near-duplicate windows land in both train and validation, so the
model wasn't really being tested on unseen data.

**Fix**: split each fault's run by time first (80% early rows for
training, 20% later rows for validation), then build windows
separately within each portion — guaranteeing no shared rows between
train and validation.

In [19]:
import numpy as np
from sklearn.model_selection import train_test_split

def build_sequences_with_run_split(df, feature_cols, window=20, step=1, val_fraction=0.2, random_state=42):
    train_seqs_X, train_seqs_y = [], []
    val_seqs_X, val_seqs_y = [], []

    for fault_num in sorted(df["faultNumber"].unique()):
        fault_data = df[df["faultNumber"] == fault_num][feature_cols].values
        n_rows = len(fault_data)

        split_point = int(n_rows * (1 - val_fraction))

        train_part = fault_data[:split_point]
        val_part = fault_data[split_point:]

        for start in range(0, len(train_part) - window + 1, step):
            train_seqs_X.append(train_part[start:start + window])
            train_seqs_y.append(fault_num)

        for start in range(0, len(val_part) - window + 1, step):
            val_seqs_X.append(val_part[start:start + window])
            val_seqs_y.append(fault_num)

    return (np.array(train_seqs_X), np.array(train_seqs_y),
            np.array(val_seqs_X), np.array(val_seqs_y))


X_train_clean, y_train_clean, X_val_clean, y_val_clean = build_sequences_with_run_split(
    train_filtered, measurement_cols, window=20
)

print("Train:", X_train_clean.shape)
print("Val:", X_val_clean.shape)

Train: (6951, 20, 52)
Val: (1467, 20, 52)


In [20]:
#scaling
n_train_samples, n_timesteps, n_features = X_train_clean.shape
n_val_samples = X_val_clean.shape[0]

scaler_clean = StandardScaler()
X_train_clean_scaled = scaler_clean.fit_transform(X_train_clean.reshape(-1, n_features))
X_train_clean_scaled = X_train_clean_scaled.reshape(n_train_samples, n_timesteps, n_features)

X_val_clean_scaled = scaler_clean.transform(X_val_clean.reshape(-1, n_features))
X_val_clean_scaled = X_val_clean_scaled.reshape(n_val_samples, n_timesteps, n_features)

label_encoder_clean = LabelEncoder()
y_train_clean_encoded = label_encoder_clean.fit_transform(y_train_clean)
y_val_clean_encoded = label_encoder_clean.transform(y_val_clean)

print("Scaled and encoded successfully")

Scaled and encoded successfully


In [21]:
# Rebuilt the model

model_lstm_v2 = keras.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(64),
    layers.Dense(19, activation="softmax")
])

model_lstm_v2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_v2 = model_lstm_v2.fit(
    X_train_clean_scaled, y_train_clean_encoded,
    validation_data=(X_val_clean_scaled, y_val_clean_encoded),
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 15s 61ms/step - accuracy: 0.4781 - loss: 1.9633 - val_accuracy: 0.5903 - val_loss: 1.3700
Epoch 2/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - accuracy: 0.8538 - loss: 0.5897 - val_accuracy: 0.7498 - val_loss: 0.8957
Epoch 3/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.9544 - loss: 0.2168 - val_accuracy: 0.7505 - val_loss: 0.9060
Epoch 4/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9856 - loss: 0.0932 - val_accuracy: 0.7478 - val_loss: 0.9348
Epoch 5/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.9919 - loss: 0.0546 - val_accuracy: 0.7485 - val_loss: 0.9507
Epoch 6/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.9990 - loss: 0.0241 - val_accuracy: 0.7532 - val_loss: 1.0151
Epoch 7/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 1.0000 - loss: 0.0120 - val_accuracy: 0.7594 - val_loss: 1.0106
Epoch 8/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 1.0000 - loss: 0.0079 - val_acc

In [22]:
from tensorflow.keras.callbacks import EarlyStopping

model_lstm_v3 = keras.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(64),
    layers.Dense(19, activation="softmax")
])

model_lstm_v3.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True
)

history_v3 = model_lstm_v3.fit(
    X_train_clean_scaled, y_train_clean_encoded,
    validation_data=(X_val_clean_scaled, y_val_clean_encoded),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 20s 71ms/step - accuracy: 0.4845 - loss: 1.9423 - val_accuracy: 0.5883 - val_loss: 1.4699
Epoch 2/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 8s 51ms/step - accuracy: 0.8458 - loss: 0.6208 - val_accuracy: 0.7082 - val_loss: 1.1254
Epoch 3/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.9376 - loss: 0.2520 - val_accuracy: 0.7491 - val_loss: 1.0373
Epoch 4/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.9732 - loss: 0.1268 - val_accuracy: 0.7532 - val_loss: 1.0282
Epoch 5/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - accuracy: 0.9908 - loss: 0.0633 - val_accuracy: 0.7519 - val_loss: 1.0813
Epoch 6/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - accuracy: 0.9919 - loss: 0.0479 - val_accuracy: 0.7819 - val_loss: 1.0418
Epoch 7/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.9996 - loss: 0.0169 - val_accuracy: 0.7730 - val_loss: 1.0860
Epoch 8/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 1.0000 - loss: 0.0106 - val_acc

In [26]:
#evaluation

n_test_samples = X_test_seq.shape[0]

X_test_seq_scaled = scaler_clean.transform(X_test_seq.reshape(-1, n_features))
X_test_seq_scaled = X_test_seq_scaled.reshape(n_test_samples, n_timesteps, n_features)

y_test_seq_encoded = label_encoder_clean.transform(y_test_seq)

test_loss, test_accuracy = model_lstm_v3.evaluate(X_test_seq_scaled, y_test_seq_encoded, verbose=0)

print(f"LSTM test accuracy: {test_accuracy:.2%}")
#print(f"XGBoost (tuned + trend) test accuracy: {accuracy_trend:.2%}")

LSTM test accuracy: 58.02%


In [24]:
y_pred_lstm_encoded = model_lstm_v3.predict(X_test_seq_scaled, verbose=0).argmax(axis=1)
y_pred_lstm = label_encoder_clean.inverse_transform(y_pred_lstm_encoded)

# Reconstruct which "starting sample" each test sequence came from, per fault
sample_positions = []
for fault_num in sorted(test_filtered["faultNumber"].unique()):
    n_rows = len(test_filtered[test_filtered["faultNumber"] == fault_num])
    n_windows = n_rows - 20 + 1
    sample_positions.extend(range(1, n_windows + 1))

sample_positions = np.array(sample_positions)

# Check accuracy specifically for windows starting before vs after sample 160
early_mask = sample_positions <= 140  # windows fully inside the pre-fault period
late_mask = sample_positions > 160

early_correct = (y_pred_lstm[early_mask] == y_test_seq[early_mask]).mean()
late_correct = (y_pred_lstm[late_mask] == y_test_seq[late_mask]).mean()

print(f"Accuracy on early (pre-fault, mislabeled) windows: {early_correct:.2%}")
print(f"Accuracy on late (genuinely faulty) windows: {late_correct:.2%}")

Accuracy on early (pre-fault, mislabeled) windows: 5.49%
Accuracy on late (genuinely faulty) windows: 68.60%


In [25]:
from feature_engineering import fix_test_labels

test_fixed_for_lstm = fix_test_labels(test)
test_filtered_fixed = test_fixed_for_lstm[~test_fixed_for_lstm["faultNumber"].isin(HARD_TO_DETECT_FAULTS)]

X_test_seq_fixed, y_test_seq_fixed = build_sequences(test_filtered_fixed, measurement_cols, window=20)

print("Fixed test sequences:", X_test_seq_fixed.shape)

Relabeled 3360 rows from faulty to normal (pre-fault period in test data)
Fixed test sequences: (18359, 20, 52)
